# Natural Disaster Chat Application

**A RAG + MCP chatbot for natural disaster data analysis and knowledge retrieval.**

This notebook composes two modules:
- **RAG Pipeline**: Hybrid retrieval (Dense + BM25 + RRF + Reranking) over PDF knowledge base
- **MCP Server**: CSV query tool over EM-DAT disaster data (1970–2021)
- **LangGraph Agent**: Routes queries to the appropriate module based on intent

## Sections
1. Setup & Configuration
2. Data Ingestion
3. Retrieval Demo
4. MCP Server Demo
5. Agent Chat
6. Evaluation
7. Visualizations
8. Architecture Diagrams

In [3]:
# --- Section 1: Setup & Configuration ---
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Ensure src/ is on the path
sys.path.insert(0, os.path.abspath("."))

from dotenv import load_dotenv
load_dotenv()

from src.config import (
    CSV_PATH, PDF_DIR, CHROMA_PERSIST_DIR, OUTPUT_DIR,
    HF_EMBEDDING_MODEL, LLM_MODEL, LLM_TEMPERATURE,
)

print(f"CSV path: {CSV_PATH} (exists: {CSV_PATH.exists()})")
print(f"PDF dir: {PDF_DIR}")
print(f"ChromaDB dir: {CHROMA_PERSIST_DIR}")
print(f"Embedding model: {HF_EMBEDDING_MODEL}")
print(f"LLM model: {LLM_MODEL}")

CSV path: /Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/data/1970-2021_DISASTERS.xlsx - emdat data.csv (exists: True)
PDF dir: /Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/data/pdfs
ChromaDB dir: /Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/chroma_db
Embedding model: BAAI/bge-m3
LLM model: gpt-4o-mini


## 2. Data Ingestion

Load CSV disaster data and optional PDF documents, then chunk and embed into ChromaDB.

In [4]:
# --- Section 2: Data Ingestion ---
from src.ingestion.loaders import load_pdfs, load_csv_as_docs
from src.ingestion.chunking import build_parent_child_chunks

# Load CSV as documents
csv_docs = load_csv_as_docs(CSV_PATH)
print(f"CSV documents loaded: {len(csv_docs)}")
print(f"Sample: {csv_docs[0].page_content[:120]}")

# Load PDFs (if available)
pdf_docs = load_pdfs(PDF_DIR)
print(f"\nPDF documents loaded: {len(pdf_docs)}")

# Combine and chunk
all_docs = csv_docs + pdf_docs
parent_chunks, child_chunks = build_parent_child_chunks(all_docs)
print(f"\nParent chunks: {len(parent_chunks)}")
print(f"Child chunks: {len(child_chunks)}")

CSV documents loaded: 14644
Sample: Year: 1970 | Country: Argentina | Disaster Type: Flood | Total Deaths: 36 | Damages (000 US$): 25000 | Location: Mendoza

PDF documents loaded: 0

Parent chunks: 14646
Child chunks: 14976


In [5]:
# --- Embed child chunks into ChromaDB ---
from src.retrieval.vectorstore import create_vectorstore, load_vectorstore, get_embeddings
from pathlib import Path

embeddings = get_embeddings(prefer_hf=True)

if Path(CHROMA_PERSIST_DIR).exists() and any(Path(CHROMA_PERSIST_DIR).iterdir()):
    print("Loading existing vectorstore...")
    vectorstore = load_vectorstore(CHROMA_PERSIST_DIR, embeddings)
else:
    print(f"Creating vectorstore with {len(child_chunks)} child chunks...")
    vectorstore = create_vectorstore(child_chunks, CHROMA_PERSIST_DIR, embeddings)

print(f"Vectorstore ready (collection count: {vectorstore._collection.count()})")

HuggingFace embeddings unavailable (No module named 'langchain_huggingface'), falling back to OpenAI


Loading existing vectorstore...
Vectorstore ready (collection count: 14976)


## 3. Retrieval Demo

Demonstrate the hybrid retriever (Dense + BM25 + RRF) and reranker.

In [6]:
# --- Section 3: Retrieval Demo ---
from src.retrieval.hybrid import HybridRetriever
from src.retrieval.reranker import rerank

# Build hybrid retriever
hybrid_retriever = HybridRetriever(vectorstore, child_chunks)

# Demo query
query = "What are the deadliest earthquakes?"
results = hybrid_retriever.retrieve(query)
print(f"Hybrid retrieval for: '{query}'")
print(f"Results: {len(results)} documents\n")

for i, doc in enumerate(results[:5]):
    print(f"  [{i+1}] {doc.page_content[:100]}...")
    print(f"      Metadata: {doc.metadata.get('country', 'N/A')} | {doc.metadata.get('year', 'N/A')}\n")

# Apply reranking
reranked = rerank(query, results)
print(f"\nAfter reranking: {len(reranked)} documents")
for i, doc in enumerate(reranked[:3]):
    print(f"  [{i+1}] {doc.page_content[:100]}...")

Hybrid retrieval for: 'What are the deadliest earthquakes?'
Results: 15 documents

  [1] Year: 1996 | Country: Peru | Disaster Type: Earthquake | Subtype: Tsunami | Total Deaths: 7 | Total ...
      Metadata: Peru | 1996

  [2] . 17 are in the North, 19 in the Northeast, 5 in the central region, 6 in the East and 1 in the Sout...
      Metadata: Thailand | 2012

  [3] Year: 2001 | Country: Peru | Disaster Type: Earthquake | Subtype: Tsunami | Total Deaths: 145 | Tota...
      Metadata: Peru | 2001

  [4] Year: 1977 | Country: Saint Vincent and the Grenadines | Disaster Type: Flood...
      Metadata: Saint Vincent and the Grenadines | 1977

  [5] Year: 1985 | Country: Mexico | Disaster Type: Earthquake | Subtype: Ground movement | Total Deaths: ...
      Metadata: Mexico | 1985


After reranking: 5 documents
  [1] Year: 2011 | Country: Japan | Disaster Type: Earthquake | Subtype: Tsunami | Total Deaths: 19846 | T...
  [2] Year: 2017 | Country: Mexico | Disaster Type: Earthquake | Subtyp

## 4. MCP Server Demo

Query the EM-DAT disaster CSV through the MCP server tool.

In [7]:
# --- Section 4: MCP Server Demo ---
from src.mcp_server.server import query_natural_disasters
import json

# Query 1: Earthquakes in Japan
result = query_natural_disasters(country="Japan", disaster_type="Earthquake")
print(f"Japan Earthquakes: {result['data']['total_matching']} records")
print(f"Timing: {result['meta']['timing_ms']:.1f}ms\n")

# Query 2: Deadliest disasters in 2010
result = query_natural_disasters(year=2010, sort_by="Total Deaths", limit=5)
print("Top 5 deadliest disasters in 2010:")
for row in result["data"]["rows"][:5]:
    print(f"  {row['Country']} - {row['Disaster Type']}: {row.get('Total Deaths', 'N/A')} deaths")

# Query 3: Year range with minimum deaths
print("\nDisasters 2000-2010 with 50,000+ deaths:")
result = query_natural_disasters(year_start=2000, year_end=2010, min_deaths=50000, sort_by="Total Deaths")
for row in result["data"]["rows"]:
    print(f"  {row['Year']} {row['Country']} - {row['Disaster Type']}: {int(row['Total Deaths']):,} deaths")

Japan Earthquakes: 45 records
Timing: 21.3ms

Top 5 deadliest disasters in 2010:
  Haiti - Earthquake: 222570.0 deaths
  Russian Federation (the) - Extreme temperature : 55736.0 deaths
  Somalia - Drought: 20000.0 deaths
  Haiti - Epidemic: 6908.0 deaths
  China - Earthquake: 2968.0 deaths

Disasters 2000-2010 with 50,000+ deaths:
  2010 Haiti - Earthquake: 222,570 deaths
  2004 Indonesia - Earthquake: 165,708 deaths
  2008 Myanmar - Storm: 138,366 deaths
  2008 China - Earthquake: 87,476 deaths
  2005 Pakistan - Earthquake: 73,338 deaths
  2010 Russian Federation (the) - Extreme temperature : 55,736 deaths


## 5. Agent Chat

The LangGraph agent routes queries to RAG (knowledge), MCP (data), or both based on intent classification.

In [8]:
# --- Section 5: Agent Chat ---
from langchain_openai import ChatOpenAI
from src.agent.graph import build_agent

# Initialize LLM
llm = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

# Build the agent with retriever and reranker
agent = build_agent(llm, retriever=hybrid_retriever, reranker_fn=rerank)
print("Agent built successfully!")

# Chat function
def chat(question: str, history: list = None):
    """Send a question to the agent and display the response."""
    if history is None:
        history = []
    result = agent.invoke({
        "question": question,
        "intent": "",
        "context": "",
        "answer": "",
        "chat_history": history,
        "sources": [],
    })
    print(f"Intent: {result['intent']}")
    print(f"\n{result['answer']}")
    return result

Agent built successfully!


In [7]:
# --- Demo: Disaster data query (routes to MCP) ---
result1 = chat("How many earthquakes occurred in Japan between 2000 and 2020? list by year")

[04/15/26 16:54:46] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513444;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513445;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[04/15/26 16:54:48] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513450;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513451;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[04/15/26 16:54:53] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513456;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513457;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Intent: disaster_data

Based on the provided data, here is the list of earthquakes that occurred in Japan between 2000 and 2020, organized by year:

- **2000**: 2 earthquakes
- **2001**: 1 earthquake
- **2003**: 3 earthquakes
- **2004**: 1 earthquake
- **2005**: 2 earthquakes
- **2007**: 3 earthquakes
- **2008**: 2 earthquakes
- **2009**: 1 earthquake (Tsunami subtype)
- **2011**: 2 earthquakes (including 1 Tsunami subtype)
- **2013**: 1 earthquake
- **2014**: 1 earthquake
- **2016**: 1 earthquake

### Summary of Earthquakes by Year:
- **2000**: 2
- **2001**: 1
- **2003**: 3
- **2004**: 1
- **2005**: 2
- **2007**: 3
- **2008**: 2
- **2009**: 1
- **2011**: 2
- **2013**: 1
- **2014**: 1
- **2016**: 1

In total, there were **17 earthquakes** recorded in Japan from 2000 to 2020.

Sources:
[Source: EM-DAT CSV, 36.2ms]


In [8]:
# --- Demo: Knowledge base query (routes to RAG) ---
result2 = chat("What causes earthquakes and how do early warning systems work?")

[04/15/26 16:55:03] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513462;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513463;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[04/15/26 16:55:04] INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=8513468;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513469;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     Hybrid retrieval: 20 dense + 20 BM25 → 15 fused                           ]8;id=8513476;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/hybrid.py\hybrid.py]8;;\:]8;id=8513477;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/hybrid.py#95\95]8;;\

[04/15/26 16:55:05] INFO     HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"  ]8;id=8513482;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513483;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\

                    INFO     Cohere reranked 15 → 5 documents                                        ]8;id=8513490;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/reranker.py\reranker.py]8;;\:]8;id=8513491;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/reranker.py#48\48]8;;\

[04/15/26 16:55:06] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513496;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513497;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[04/15/26 16:55:07] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513502;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513503;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Intent: mixed

I don't have enough information to answer that question accurately. Could you rephrase or provide more details?

Sources:
[Source: 1970-2021_DISASTERS.xlsx - emdat data.csv]
[Source: EM-DAT CSV, 38.2ms]


In [9]:
# --- Demo: Mixed query (routes to both RAG + MCP) ---
result3 = chat("Why was the 2010 Haiti earthquake so deadly compared to other earthquakes?")

[04/15/26 16:55:20] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513508;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513509;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

                    INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=8513514;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513515;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     Hybrid retrieval: 20 dense + 20 BM25 → 15 fused                           ]8;id=8513520;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/hybrid.py\hybrid.py]8;;\:]8;id=8513521;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/hybrid.py#95\95]8;;\

[04/15/26 16:55:21] INFO     HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 200 OK"  ]8;id=8513526;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513527;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\

                    INFO     Cohere reranked 15 → 5 documents                                        ]8;id=8513532;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/reranker.py\reranker.py]8;;\:]8;id=8513533;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/reranker.py#48\48]8;;\

[04/15/26 16:55:22] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513538;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513539;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

[04/15/26 16:55:27] INFO     HTTP Request: POST https://api.openai.com/v1/chat/completions          ]8;id=8513544;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=8513545;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             "HTTP/1.1 200 OK"                                                                     

Intent: mixed

The 2010 Haiti earthquake was particularly deadly due to several interrelated factors:

1. **High Population Density**: The earthquake struck near Port-au-Prince, the capital city, which has a high population density. This meant that a large number of people were in close proximity to the epicenter when the earthquake occurred.

2. **Poor Infrastructure**: Haiti's infrastructure was already weak prior to the earthquake, with many buildings constructed without adherence to seismic standards. The collapse of poorly built structures contributed significantly to the high mortality rate.

3. **Lack of Preparedness**: Haiti had limited disaster preparedness and response capabilities. The country was still recovering from years of political instability and economic challenges, which hindered effective emergency management.

4. **Magnitude and Depth**: The earthquake had a magnitude of 7.0 and was shallow (approximately 13 km deep), which amplified its destructive impact on the 

## 6. Evaluation

Evaluate using the golden dataset (50 Q&A pairs) and compare retrieval strategies.

In [1]:
# --- Section 6: Evaluation ---
from src.evaluation.evaluator import (
    load_golden_dataset, evaluate_csv_query_correctness, run_full_evaluation
)
from src.evaluation.dashboard import render_metrics_table, render_csv_correctness_chart

# Load golden dataset
golden = load_golden_dataset()
print(f"Golden dataset: {len(golden)} items")

# CSV query correctness
csv_eval = evaluate_csv_query_correctness(golden)
print(f"\nCSV Query Correctness: {csv_eval['correct']}/{csv_eval['total']} = {csv_eval['accuracy']:.1%}")

# Display correctness chart
fig = render_csv_correctness_chart(csv_eval["details"])
fig.show()

Golden dataset: 50 items

CSV Query Correctness: 20/20 = 100.0%


In [9]:
# --- Strategy Comparison ---
from src.evaluation.strategy_comparison import compare_strategies, render_comparison_report

strategy_results = compare_strategies(vectorstore, child_chunks, reranker_fn=rerank)

print("Strategy Comparison — Hit Rate@5:")
for name, vals in strategy_results.items():
    print(f"  {name}: {vals['hit_rate_at_5']:.4f}")

# Render comparison charts
table_fig, chart_fig = render_comparison_report(strategy_results)
table_fig.show()
chart_fig.show()

[04/15/26 16:58:08] INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=16674430;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16674431;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=16674436;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16674437;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=16674442;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16674443;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=16674448;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16674449;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200  ]8;id=16674454;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16674455;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

...

[04/15/26 16:58:59] INFO     HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 429 Too  ]8;id=16675406;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16675407;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             Many Requests"                                                                        

[04/15/26 16:59:01] INFO     HTTP Request: POST https://api.cohere.com/v2/rerank "HTTP/1.1 429 Too  ]8;id=16675412;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=16675413;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             Many Requests"                                                                        

                    WARNING  Cohere reranking failed (headers: {'access-control-expose-headers':     ]8;id=16675418;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/reranker.py\reranker.py]8;;\:]8;id=16675419;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/retrieval/reranker.py#52\52]8;;\
                             'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform,               
                             must-revalidate, private, max-age=0', 'content-encoding': 'gzip',                     
                             'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970                      
                             00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding',                
                             'x-accel-expires': '0', 'x-debug-trace-id':                                           
                             '431583054c550ce5e8a84575dc626eeb', 'date': 'Wed, 15 Apr 2026 13:59:01                
                             GMT', 'x-envoy-upstream-service-time': '1', 'server': 'envoy', 'via':                 
                             '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443";                         
                             ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 429, body:                 
                             {'id': '3f2f70ab-1633-4f48-871c-a4f5767a060b', 'message': "You are                    
                             using a Trial key, which is limited to 10 API calls / minute. You can                 
                             continue to use the Trial key for free or upgrade to a Production key                 
                             with higher rate limits at 'https://dashboard.cohere.com/api-keys'.                   
                             Contact us on 'https://discord.gg/XW44jPfYJu' or email us at                          
                             support@cohere.com with any questions"}) — falling back to top-5 by                   
                             position                                                                              

                    INFO     Hybrid+Rerank: 0.6000                                       ]8;id=16675425;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/evaluation/strategy_comparison.py\strategy_comparison.py]8;;\:]8;id=16675426;file:///Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/src/evaluation/strategy_comparison.py#104\104]8;;\

Strategy Comparison — Hit Rate@5:
  dense_only: 0.6000
  bm25_only: 0.6500
  hybrid: 0.6000
  hybrid_rerank: 0.6000


## 7. Visualizations

Interactive charts from the EM-DAT disaster dataset.

In [10]:
# --- Section 7: Visualizations ---
from src.visualization.charts import create_all_charts

charts = create_all_charts()
for name, fig in charts.items():
    print(f"Rendering: {name}")
    fig.show()

Rendering: disaster_type_pie


Rendering: disasters_by_year


Rendering: top_countries


Rendering: deaths_vs_affected


Rendering: monthly_heatmap


In [11]:
# --- Network Diagram ---
from src.visualization.diagrams import country_disaster_network

network_fig = country_disaster_network()
network_fig.show()

In [12]:
# --- Generate HTML Report ---
from src.visualization.report import generate_html_report

report_path = generate_html_report()
print(f"HTML report saved to: {report_path}")
print(f"File size: {report_path.stat().st_size / 1024:.0f} KB")

HTML report saved to: /Users/Dzmitry_Luhavy/Documents/SASAI/FINAL/dl-ai-sas-chat-rag-mcp/output/report.html
File size: 5159 KB


## 8. Architecture Diagrams

### System Architecture

```mermaid
graph TD
    A[User] --> B[Jupyter Notebook Chat UI]
    B --> C[LangGraph Agent - ReAct Routing]
    C --> D{Intent Classification}
    D -->|disaster_data| E[MCP Server]
    D -->|knowledge_base| F[RAG Pipeline]
    D -->|mixed| G[Both Paths]
    D -->|general| H[Direct LLM Response]

    E --> I[CSV Query Tool - Pandas]
    I --> J[(EM-DAT CSV 1970-2021)]

    F --> K[Hybrid Retriever]
    K --> L[Dense Search - ChromaDB]
    K --> M[Sparse Search - BM25]
    L --> N[Reciprocal Rank Fusion]
    M --> N
    N --> O[Cohere Reranker]

    G --> E
    G --> F

    O --> P[LLM Generation - GPT-4o-mini]
    I --> P
    H --> P
    P --> Q[Response with Citations]
    Q --> B

    style A fill:#3498db,color:#fff
    style C fill:#2ecc71,color:#fff
    style E fill:#e74c3c,color:#fff
    style F fill:#9b59b6,color:#fff
    style J fill:#f39c12,color:#fff
    style L fill:#1abc9c,color:#fff
    style P fill:#e67e22,color:#fff
```

### RAG Pipeline Detail

```mermaid
graph LR
    A[PDF Documents] --> B[PyPDF Loader]
    C[CSV Data] --> D[Row-to-Doc Converter]
    B --> E[Parent Chunker 2048 chars]
    D --> E
    E --> F[Child Chunker 512 chars]
    F --> G[HuggingFace Embeddings BGE-M3]
    G --> H[(ChromaDB Vector Store)]

    I[User Query] --> J[Dense Search top-20]
    I --> K[BM25 Sparse Search top-20]
    H --> J
    J --> L[RRF Fusion top-15]
    K --> L
    L --> M[Cohere Reranker top-5]
    M --> N[LLM Generation]
    N --> O[Answer + Citations]

    style H fill:#3498db,color:#fff
    style L fill:#2ecc71,color:#fff
    style M fill:#e74c3c,color:#fff
    style N fill:#e67e22,color:#fff
```

## 9. Streamlit Chat UI

Launch the interactive chat interface powered by the same LangGraph agent.
Once running, open [http://localhost:8501](http://localhost:8501) in your browser.

In [1]:
# 🚀 Launch the Streamlit chat app (opens in browser at http://localhost:8501)
import subprocess, sys, os

app_dir = os.path.dirname(os.path.abspath("streamlit_app.py"))
_streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "streamlit_app.py", "--server.port", "8501"],
    cwd=app_dir,
)
print(f"App started → http://localhost:8501  (PID {_streamlit_proc.pid})")

App started → http://localhost:8501  (PID 44204)



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://10.22.213.14:8501

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            


HuggingFace embeddings unavailable (No module named 'langchain_huggingface'), falling back to OpenAI


In [2]:
# 🛑 Stop the Streamlit app
try:
    _streamlit_proc.terminate()
    _streamlit_proc.wait(timeout=5)
    print(f"App stopped (PID {_streamlit_proc.pid})")
except NameError:
    import subprocess
    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    print("App stopped")

  Stopping...
App stopped (PID 44204)
